# 04 — Tournament Simulation (Bayesian Update + Monte Carlo)

**Purpose:** Apply the Bayesian tournament update layer, then execute the 10,000-simulation
Monte Carlo engine. Covers both design-doc notebooks 04 (Bayesian update) and 05 (Monte Carlo),
consolidated here because the update is applied inside the simulation, not as a separate
pre-computation step.

**Inputs**
- `outputs/group_stage_model.joblib`
- `outputs/knockout_model.joblib`
- `outputs/penalty_model.joblib`
- `outputs/team_features_freeze.parquet`

**Outputs**
- `outputs/bayesian_update_table.csv`
- `outputs/win_probabilities.csv`
- `outputs/simulation_log.parquet`
- `outputs/bracket_simulation_summary.csv`
- `outputs/charts/update_shifts.png`
- `outputs/charts/win_probability_chart.png`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.leakage_guard import check_no_synthetic_data, check_freeze_date, LeakageError
from src.name_map import CANONICAL_48, canonicalize
from src.models import BayesianTournamentUpdater, BAYESIAN_ALPHA, PRETOURNAMENT_FEATURES
from src.simulation import (
    MonteCarloEngine, run_tournament_simulation,
    assert_simulation_inputs_frozen, compute_md3_proba,
    build_md3_schedule,
)
from src.features import load_ds16, load_ds17, load_ds1

DATA_ROOT = PROJECT_ROOT
ARC_BASE  = DATA_ROOT / "archive.zip"
ARC2      = DATA_ROOT / "archive (2).zip"
OUTPUTS   = PROJECT_ROOT / "outputs"
CHARTS    = OUTPUTS / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## Cell 2 — Load models and features

In [ ]:
group_model   = joblib.load(OUTPUTS / "group_stage_model.joblib")
knockout_model= joblib.load(OUTPUTS / "knockout_model.joblib")
penalty_model = joblib.load(OUTPUTS / "penalty_model.joblib")
team_features = pd.read_parquet(OUTPUTS / "team_features_freeze.parquet")

ds16 = load_ds16(ARC_BASE)
ds17 = load_ds17(ARC_BASE)
ds1  = load_ds1(ARC2)

# Re-run leakage guard on the feature table
check_no_synthetic_data(team_features)
assert_simulation_inputs_frozen(team_features)
print("✓ All simulation input leakage guards passed")

print(f"\nTeam features: {team_features.shape}")
print(f"DS16 (bracket): {ds16.shape}")
print(f"DS17 (teams):   {ds17.shape}")

## Cell 3 — Bayesian update: compute form multipliers

In [ ]:
updater = BayesianTournamentUpdater(alpha=BAYESIAN_ALPHA)
print(f"BayesianTournamentUpdater(alpha={BAYESIAN_ALPHA})")

# Compute form multipliers for all teams using in-tournament data
multiplier_rows = []
for team in sorted(CANONICAL_48):
    if team not in team_features.index:
        continue
    feats = team_features.loc[team]
    try:
        # Elo-expected points/GD as baseline for multiplier computation
        elo_we = float(feats.get("elo_win_expectancy", 0.5))
        elo_expected_pts = elo_we * 3.0
        elo_expected_gd  = (elo_we - 0.5) * 2.0

        multiplier = updater.compute_form_multiplier(
            feats, elo_expected_pts=elo_expected_pts, elo_expected_gd=elo_expected_gd
        )
        multiplier_rows.append({"team": team, "form_multiplier": multiplier})
    except Exception:
        multiplier_rows.append({"team": team, "form_multiplier": None})

mult_df = pd.DataFrame(multiplier_rows).dropna(subset=["form_multiplier"])
mult_df = mult_df.sort_values("form_multiplier", ascending=False)

print("\nForm multipliers (top 10 positive signals):")
display(mult_df.head(10))
print("\nForm multipliers (bottom 5 negative signals):")
display(mult_df.tail(5))

# Design-spec assertions
if not mult_df.empty:
    assert mult_df["form_multiplier"].between(0.5, 2.0).all(), (
        "All multipliers must be in [0.5, 2.0]"
    )
    print("\n✓ All form multipliers within [0.5, 2.0]")

## Cell 4 — Bayesian update: before / after probability table

In [ ]:
feature_cols = [c for c in PRETOURNAMENT_FEATURES if c in team_features.columns]

update_rows = []
for team in sorted(CANONICAL_48):
    if team not in team_features.index:
        continue
    feats = team_features.loc[team]

    # Prior win probability from model (vs neutral average opponent)
    try:
        prior_vec = np.array([1/3, 1/3, 1/3])  # flat prior if model not available
        if feature_cols:
            X = pd.DataFrame([feats[feature_cols]])
            proba = group_model.predict_proba(X)
            prior_vec = proba[0]

        elo_we = float(feats.get("elo_win_expectancy", 0.5))
        posterior_vec = updater.apply_update(
            prior_vec, feats,
            elo_expected_pts=elo_we * 3.0,
            elo_expected_gd=(elo_we - 0.5) * 2.0,
        )

        p_win_prior     = float(prior_vec[2])     # WIN class index
        p_win_posterior = float(posterior_vec[2])

        update_rows.append({
            "team":               team,
            "p_win_prior":        round(p_win_prior, 4),
            "p_win_posterior":    round(p_win_posterior, 4),
            "shift":              round(p_win_posterior - p_win_prior, 4),
        })
    except Exception as e:
        update_rows.append({"team": team, "p_win_prior": None,
                             "p_win_posterior": None, "shift": None})

update_df = pd.DataFrame(update_rows).dropna()
print("5 largest positive shifts (improving teams):")
display(update_df.sort_values("shift", ascending=False).head(5))
print("\n5 largest negative shifts (underperforming teams):")
display(update_df.sort_values("shift").head(5))

## Cell 5 — Visualise Bayesian update shifts

In [ ]:
plot_df = update_df.sort_values("shift")
colors  = ["#2ecc71" if s > 0.01 else "#e74c3c" if s < -0.01 else "#95a5a6"
           for s in plot_df["shift"]]

fig, ax = plt.subplots(figsize=(9, max(6, len(plot_df) * 0.22)))
ax.barh(plot_df["team"], plot_df["shift"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"Win probability shift (α = {BAYESIAN_ALPHA})")
ax.set_title("Bayesian Tournament Update — Probability Shifts (MD1 + MD2 signal)")
plt.tight_layout()

out_path = CHARTS / "update_shifts.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

## Cell 6 — α sensitivity table

In [ ]:
# Design spec: show what happens at α = 0.0, 0.17, 0.30
sensitivity_rows = []
for team in sorted(CANONICAL_48)[:10]:  # top 10 by Elo for clarity
    if team not in team_features.index:
        continue
    feats = team_features.loc[team]

    try:
        X = pd.DataFrame([feats[feature_cols]]) if feature_cols else None
        prior_vec = group_model.predict_proba(X)[0] if X is not None else np.array([1/3,1/3,1/3])
        elo_we = float(feats.get("elo_win_expectancy", 0.5))

        row = {"team": team}
        for alpha in [0.0, 0.17, 0.30]:
            u = BayesianTournamentUpdater(alpha=alpha)
            posterior = u.apply_update(prior_vec.copy(), feats,
                                        elo_expected_pts=elo_we*3,
                                        elo_expected_gd=(elo_we-0.5)*2)
            row[f"p_win_alpha={alpha}"] = round(float(posterior[2]), 4)
        sensitivity_rows.append(row)
    except Exception:
        pass

if sensitivity_rows:
    print(f"α sensitivity (conservative α=0.17 chosen to avoid over-reaction):")
    pd.DataFrame(sensitivity_rows)

## Cell 7 — Test run: 1,000 simulations

In [ ]:
md3_schedule = build_md3_schedule(ds16, ds17)

print("Running 1,000 simulations (test run)...")
t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results_1k = run_tournament_simulation(
        n_simulations=1_000,
        group_model=group_model,
        knockout_model=knockout_model,
        penalty_model=penalty_model,
        team_features=team_features,
        md3_schedule=md3_schedule,
        ds16=ds16,
        ds17=ds17,
        annex_c_path=DATA_ROOT / "third_place_annex_c.csv",
        seed=2026,
    )

elapsed = time.time() - t0
print(f"1,000 simulations completed in {elapsed:.1f}s")
print(f"Estimated time for 10,000 simulations: {elapsed*10:.0f}s ({elapsed*10/60:.1f} min)")

# Invariant checks on 1k results
if hasattr(results_1k, "win_probabilities"):
    wp = results_1k.win_probabilities
    total = sum(wp.values()) if isinstance(wp, dict) else wp.sum()
    assert abs(total - 1.0) < 1e-4, f"Win probs must sum to 1.0, got {total}"
    print(f"\n✓ Win probabilities sum = {total:.6f}")

    if isinstance(wp, dict):
        top5 = sorted(wp.items(), key=lambda x: -x[1])[:5]
    else:
        top5 = wp.sort_values(ascending=False).head(5)
    print("\nTop 5 (1k run):")
    print(top5)

## Cell 8 — Full 10,000-simulation run

In [ ]:
print("Running full 10,000 simulations...")
t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results = run_tournament_simulation(
        n_simulations=10_000,
        group_model=group_model,
        knockout_model=knockout_model,
        penalty_model=penalty_model,
        team_features=team_features,
        md3_schedule=md3_schedule,
        ds16=ds16,
        ds17=ds17,
        annex_c_path=DATA_ROOT / "third_place_annex_c.csv",
        seed=2026,
    )

elapsed = time.time() - t0
print(f"10,000 simulations completed in {elapsed:.1f}s")

# Core invariants
if hasattr(results, "win_probabilities"):
    wp = results.win_probabilities
    if isinstance(wp, dict):
        wp_series = pd.Series(wp)
    else:
        wp_series = wp

    assert abs(wp_series.sum() - 1.0) < 1e-4, f"Sum of win probs = {wp_series.sum()}"
    assert (wp_series >= 0).all(), "Negative win probabilities found"
    print(f"\n✓ Win probabilities sum = {wp_series.sum():.6f}")
    print(f"✓ No negative probabilities")

    print("\nTop 10 tournament win probabilities:")
    display(wp_series.sort_values(ascending=False).head(10).rename("p_win").reset_index())

## Cell 9 — Confidence intervals and bracket summary

In [ ]:
if hasattr(results, "win_probabilities"):
    wp_series = pd.Series(results.win_probabilities) if isinstance(
        results.win_probabilities, dict) else results.win_probabilities

    n = 10_000
    z = 1.645  # 90% CI
    ci_df = pd.DataFrame({
        "team":    wp_series.index,
        "p_win":   wp_series.values,
        "ci_lo":   (wp_series - z * np.sqrt(wp_series * (1 - wp_series) / n)).clip(0).values,
        "ci_hi":   (wp_series + z * np.sqrt(wp_series * (1 - wp_series) / n)).clip(0, 1).values,
    })
    ci_df["ci_width"] = ci_df["ci_hi"] - ci_df["ci_lo"]
    ci_df = ci_df.sort_values("p_win", ascending=False).reset_index(drop=True)

    wide_ci = ci_df[ci_df["ci_width"] > 0.05]
    print(f"Teams with CI width > 5pp: {len(wide_ci)}")
    print("\nFull win probability table with 90% CI:")
    display(ci_df)

if hasattr(results, "simulation_log") and results.simulation_log is not None:
    log = results.simulation_log
    assert len(log) == 10_000, f"Expected 10,000 simulation log rows, got {len(log)}"
    print(f"\n✓ Simulation log has {len(log)} rows")

## Cell 10 — Save all outputs

In [ ]:
# Bayesian update table
update_path = OUTPUTS / "bayesian_update_table.csv"
update_df.to_csv(update_path, index=False)
print(f"Saved → {update_path}")

# Win probabilities (with CI)
if 'ci_df' in dir():
    wp_path = OUTPUTS / "win_probabilities.csv"
    ci_df.to_csv(wp_path, index=False)
    print(f"Saved → {wp_path}")

# Simulation log
if hasattr(results, "simulation_log") and results.simulation_log is not None:
    log_path = OUTPUTS / "simulation_log.parquet"
    results.simulation_log.to_parquet(log_path, index=False)
    print(f"Saved → {log_path}")

# Bracket summary
if hasattr(results, "bracket_summary") and results.bracket_summary is not None:
    bs_path = OUTPUTS / "bracket_simulation_summary.csv"
    results.bracket_summary.to_csv(bs_path, index=False)
    print(f"Saved → {bs_path}")

# Win probability chart
if 'ci_df' in dir() and not ci_df.empty:
    fig, ax = plt.subplots(figsize=(9, max(6, len(ci_df) * 0.22)))
    ax.barh(ci_df["team"], ci_df["p_win"], xerr=[
        ci_df["p_win"] - ci_df["ci_lo"],
        ci_df["ci_hi"] - ci_df["p_win"]
    ], capsize=3, color="#3498db", alpha=0.8)
    ax.set_xlabel("Tournament win probability")
    ax.set_title(
        "FIFA World Cup 2026 — Win Probabilities\n"
        "Based on 10,000 Monte Carlo simulations, Matchday 2 freeze (2026-06-23)"
    )
    plt.tight_layout()
    chart_path = CHARTS / "win_probability_chart.png"
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {chart_path}")

print("\n✓ All simulation outputs saved")